# HomeBot Qwen3.5 Fine-tune (native NVIDIA GPU, bf16 LoRA)

Companion to [`unsloth_qwen3_5_4b_homebot.ipynb`](./unsloth_qwen3_5_4b_homebot.ipynb) (the Colab T4 notebook). This version targets **local / self-hosted NVIDIA GPUs** (Ampere A100/A10, Ada L4/RTX 4090/RTX 6000 Ada, Hopper H100/H200, Blackwell B200/RTX Pro 6000) and takes advantage of:

- **Native bf16** on Ampere and newer (no fp16 fallback, no dtype mismatch drama).
- **Flash Attention 2/3** (picked up automatically by Unsloth when installed).
- **`FastLanguageModel`** (simpler path than the VLM loader) -- our data is 100% text so we skip the vision tower entirely.
- **Larger batch sizes** auto-tuned to detected free VRAM.
- **Modern `datasets`** so `load_dataset()` Just Works -- no raw-parquet fallback needed.

Everything else matches the Colab notebook: same data repo (`kanakjr/homebot-qwen3.5`), same chat template (`qwen3-instruct`), same `train_on_responses_only` loss mask, same GGUF export for Ollama, same optional HF Hub push.

## Model size toggle

```python
MODEL_SIZE = "4B"   # "2B" or "4B"
```

| Size | VRAM (bf16 LoRA, batch 8) | ~2 epoch wall-clock on RTX 6000 Ada (48 GB) | Role |
|---|---|---|---|
| **Qwen3.5-2B** | ~6 GB  | ~3 min | smoke test / fast iteration |
| **Qwen3.5-4B** | ~12 GB | ~8 min | final ship build |

On B200 (~180 GB HBM3e) expect roughly 4-5x faster wall-clock than RTX 6000 Ada.

## Inputs required

1. An NVIDIA GPU with `nvidia-smi` working and >=16 GB free VRAM.
2. HF **write** token pasted into step 0 below.
3. `MODEL_SIZE` set to `"2B"` or `"4B"` (step 0).

Outputs land in `./{BUILD_TAG}/` (GGUF) and `./{BUILD_TAG}.Modelfile`, ready for `ollama create`.


## 0. Configuration -- set your HF token + pick a model size

All knobs live in this one cell. Set them once, hit Run All, come back in 5-15 minutes to a GGUF + Modelfile ready to `ollama create`.


In [ ]:
import os
import torch

# --- REQUIRED: paste your HF write token here -----------------------------
HF_TOKEN = ""  # e.g. "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
# --------------------------------------------------------------------------

# --- Model size toggle ----------------------------------------------------
# "2B"  -> ~6  GB VRAM, ~3 min / 2 epochs on RTX 6000 Ada.
# "4B"  -> ~12 GB VRAM, ~8 min / 2 epochs on RTX 6000 Ada; the ship build.
MODEL_SIZE = "4B"   # "2B" or "4B"

_MODEL_REGISTRY = {
    "2B": {
        "name":    "unsloth/Qwen3.5-2B",
        "gguf":    "kanakjr/homebot-qwen3.5-2b-gguf",
        "max_seq": 4096,
    },
    "4B": {
        "name":    "unsloth/Qwen3.5-4B",
        "gguf":    "kanakjr/homebot-qwen3.5-4b-gguf",
        "max_seq": 4096,
    },
}
assert MODEL_SIZE in _MODEL_REGISTRY, f"MODEL_SIZE must be one of {list(_MODEL_REGISTRY)}"
_m = _MODEL_REGISTRY[MODEL_SIZE]
MODEL_NAME     = _m["name"]
MAX_SEQ_LENGTH = _m["max_seq"]

# BUILD_TAG is threaded through every artifact path (LoRA dir, GGUF dir,
# Modelfile filename, Ollama model name) so 2B and 4B runs don't overwrite
# each other. e.g. "homebot-qwen3_5-2b", "homebot-qwen3_5-4b".
BUILD_TAG = f"homebot-qwen3_5-{MODEL_SIZE.lower()}"

# --- Dataset + (optional) model repo names --------------------------------
HUB_DATASET_REPO = "kanakjr/homebot-qwen3.5"
HUB_MODEL_REPO   = _m["gguf"]

# --- Training knobs -------------------------------------------------------
REAL_OVERSAMPLE = 4       # step 6.5: how many times each real Telegram row is repeated
LORA_R = 16; LORA_ALPHA = 16     # bump to 32/64 if under-fitting
NUM_EPOCHS   = 2
LEARNING_RATE = 2e-4

# BATCH_SIZE is auto-selected from detected free VRAM in step 2.
# Override here if you want an explicit value.
BATCH_SIZE = None         # None = auto; else int (e.g. 2, 4, 8, 16)
GRAD_ACCUM = 1            # keep 1 unless OOM

# --- Auth setup -----------------------------------------------------------
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

assert HF_TOKEN, "HF_TOKEN is empty -- paste your HF write token above."
print(f"[config] HF_TOKEN set (len={len(HF_TOKEN)})")
print(f"[config] MODEL_SIZE   = {MODEL_SIZE}")
print(f"[config] base model   = {MODEL_NAME}")
print(f"[config] max seq len  = {MAX_SEQ_LENGTH}")
print(f"[config] build tag    = {BUILD_TAG}")
print(f"[config] dataset repo = {HUB_DATASET_REPO}")
print(f"[config] gguf   repo  = {HUB_MODEL_REPO}")


## 1. Install Unsloth + dependencies

Idempotent: skipped quickly if `unsloth` is already importable. No Colab-specific hacks (no `torchcodec` uninstall, no `uv`). Unsloth's installer auto-detects your CUDA + PyTorch combination.


In [ ]:
%%capture
import importlib.util, subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

if importlib.util.find_spec("unsloth") is None:
    _pip("unsloth")
    _pip("unsloth_zoo")

# Pin matching versions for Qwen3.5-specific fixes; upgrade if present.
_pip(
    "--upgrade",
    "trl==0.22.2",
    "transformers==5.2.0",
    "datasets",
    "huggingface_hub",
    "bitsandbytes",
)

# Optional: flash-attn for ~15-20% speedup on Ampere+. Wheels are big; skip
# silently if unavailable for your torch/cuda combo.
try:
    _pip("flash-attn", "--no-build-isolation")
except subprocess.CalledProcessError:
    print("[install] flash-attn install failed; falling back to xformers")

import unsloth, trl, transformers, datasets
print(f"unsloth      : {unsloth.__version__}")
print(f"trl          : {trl.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")


## 2. Verify GPU + pick batch size

Dumps detected GPU, compute capability, free VRAM, and bf16 support. If `BATCH_SIZE` was left at `None` in step 0, we pick a sensible default based on free VRAM. Fails fast if no NVIDIA GPU is visible.


In [ ]:
assert torch.cuda.is_available(), (
    "No CUDA GPU visible. Run `nvidia-smi` in the same shell that launched "
    "Jupyter and restart the kernel."
)

dev  = torch.cuda.current_device()
prop = torch.cuda.get_device_properties(dev)
free_bytes, total_bytes = torch.cuda.mem_get_info(dev)
free_gb  = free_bytes  / (1024 ** 3)
total_gb = total_bytes / (1024 ** 3)
bf16_ok  = torch.cuda.is_bf16_supported()

print(f"[gpu] device           : {prop.name}")
print(f"[gpu] compute capab    : {prop.major}.{prop.minor}")
print(f"[gpu] total / free VRAM: {total_gb:.1f} / {free_gb:.1f} GB")
print(f"[gpu] bf16 supported   : {bf16_ok}")

if not bf16_ok:
    print("[gpu] WARNING: no native bf16; training will use fp16 (slower, more memory).")

# Auto-select BATCH_SIZE if caller left it at None. These numbers are
# conservative; if you have slack after the first training step, bump
# BATCH_SIZE up in step 0 and re-run.
if BATCH_SIZE is None:
    if MODEL_SIZE == "2B":
        BATCH_SIZE = 16 if free_gb >= 40 else 8 if free_gb >= 20 else 4 if free_gb >= 10 else 2
    else:  # 4B
        BATCH_SIZE = 16 if free_gb >= 80 else 8 if free_gb >= 40 else 4 if free_gb >= 20 else 2
    print(f"[gpu] auto-selected BATCH_SIZE = {BATCH_SIZE}")
else:
    print(f"[gpu] using explicit BATCH_SIZE = {BATCH_SIZE}")

TRAIN_DTYPE = torch.bfloat16 if bf16_ok else torch.float16
print(f"[gpu] training dtype   : {TRAIN_DTYPE}")


## 3. Load the HomeBot dataset

Pulls directly from the HF Hub repo set in step 0. On a modern `datasets` (>=3.0) this Just Works -- no raw-parquet fallback or JSON-decode shim needed (those are Colab-specific workarounds).


In [ ]:
from collections import Counter
from datasets import load_dataset

ds = load_dataset(HUB_DATASET_REPO, token=HF_TOKEN)
print(ds)

train_sources = Counter(r.get("source", "?") for r in ds["train"])
val_sources   = Counter(r.get("source", "?") for r in ds["validation"])
print(f"train sources: {dict(train_sources)}")
print(f"val   sources: {dict(val_sources)}")

print("\nTrain example keys:", list(ds["train"][0].keys()))
print("First user turn:", next(
    (m["content"][:140] for m in ds["train"][0]["messages"] if m["role"] == "user"),
    "(no user turn found)",
))


## 4. Load Qwen3.5 in bf16 LoRA mode

On Ampere+ the straightforward path is `FastLanguageModel` with `dtype=torch.bfloat16`. No VLM loader needed (that was a T4-only workaround), and Flash Attention 2 is picked up automatically when installed.


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = TRAIN_DTYPE,
    load_in_4bit    = False,
    use_gradient_checkpointing = "unsloth",
)


## 5. Attach LoRA adapters

`r=16, lora_alpha=16` is the Unsloth-recommended starting point. On a 48 GB card feel free to bump both to 32 or 64 if the training loss plateaus early; on B200 you can easily go to 128 without VRAM concerns.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 6. Apply the qwen3-instruct chat template

`qwen3-instruct` expects ChatML (`<|im_start|>role\ncontent<|im_end|>`). Our training JSONL already has `role=system/user/assistant/tool` with OpenAI-style `tool_calls`, so `apply_chat_template` handles everything. Thinking mode is **disabled by default** on small Qwen3.5 models -- exactly what we want for deterministic tool-calling.


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)


## 6.5 Oversample real Telegram rows

The merge step flagged a ~13:1 synthetic:real ratio. Real Telegram chats are the most authentic signal for how the user actually talks (short, elliptical, often no greeting), so we duplicate them `REAL_OVERSAMPLE` times inside the train split. This is a "poor-man's sample weights" -- simpler than trainer-level weighting, and it plays cleanly with `train_on_responses_only`.

Rule of thumb: set `REAL_OVERSAMPLE` so the effective ratio ends up ~3:1 or tighter. With 204 synthetic / 16 real in train, `REAL_OVERSAMPLE=4` gives 64 real, i.e. ~3.2:1.


In [ ]:
from datasets import concatenate_datasets

real_rows  = ds["train"].filter(lambda r: r.get("source") == "telegram")
synth_rows = ds["train"].filter(lambda r: r.get("source") != "telegram")

if len(real_rows) == 0:
    print("[oversample] no real rows found; skipping")
    train_balanced = ds["train"]
else:
    real_repeated  = concatenate_datasets([real_rows] * REAL_OVERSAMPLE)
    train_balanced = concatenate_datasets([synth_rows, real_repeated]).shuffle(seed=3407)
    print(
        f"[oversample] real rows {len(real_rows)} -> {len(real_repeated)} "
        f"({REAL_OVERSAMPLE}x); synth unchanged at {len(synth_rows)}; "
        f"final train size = {len(train_balanced)}"
    )

ds_train = train_balanced
ds_val   = ds["validation"]
print(f"train size={len(ds_train)}, val size={len(ds_val)}")


## 7. Render each conversation through the chat template

Each row's `messages` list becomes a single `text` string that `SFTTrainer` tokenizes. We drop all original columns via `remove_columns=...` to avoid the "Could not infer dtype of dict" error from the default data collator.


In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts  = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": texts}

ds_train = ds_train.map(
    formatting_prompts_func, batched=True, remove_columns=ds_train.column_names
)
ds_val = ds_val.map(
    formatting_prompts_func, batched=True, remove_columns=ds_val.column_names
)

print("Train rows:", len(ds_train))
print("First rendered example (first 400 chars):")
print(ds_train[0]["text"][:400])


## 8. Configure SFTTrainer

Native bf16 on Ampere+ means `bf16=True` just works -- no `fp16=True`/`bf16=False` dance. Flash Attention 2 is picked up automatically if installed.

`per_device_train_batch_size = BATCH_SIZE` was auto-selected in step 2 based on detected VRAM; override by setting `BATCH_SIZE` explicitly in step 0.


In [ ]:
from trl import SFTTrainer, SFTConfig

USE_BF16 = (TRAIN_DTYPE == torch.bfloat16)
USE_FP16 = not USE_BF16

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = ds_train,
    eval_dataset    = ds_val,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        output_dir = f"outputs/{BUILD_TAG}",
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio = 0.03,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        bf16 = USE_BF16,
        fp16 = USE_FP16,
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 20,
        save_strategy = "no",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)


## 9. `train_on_responses_only` -- loss ONLY on assistant + tool_calls

The system prompt and user turns are masked out so the model spends its gradient budget learning to produce correct tool calls + terse assistant summaries, rather than memorising the prompt.


In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)


## 10. Memory snapshot before training


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory       = round(gpu_stats.total_memory       / 1024 / 1024 / 1024, 3)
print(f"{gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


## 11. Train

On RTX 6000 Ada (48 GB), 2 epochs of Qwen3.5-4B bf16 LoRA with batch size 8 is ~8 min. On B200 expect ~2 min. If you see persistent OOM here, lower `BATCH_SIZE` or raise `GRAD_ACCUM` in step 0.


In [ ]:
trainer_stats = trainer.train()


## 12. Memory snapshot after training


In [ ]:
used_memory          = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage      = round(used_memory          / max_memory * 100, 3)
lora_percentage      = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
print(f"Peak reserved memory          = {used_memory} GB.")
print(f"Peak reserved memory for LoRA = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max = {used_percentage} %.")
print(f"Peak LoRA memory %            = {lora_percentage} %.")


## 13. Sanity inference on held-out HomeBot prompts

Three prompts covering: a straightforward light control, a Spotify playback request, and a multi-entity status query. `FastLanguageModel.for_inference(model)` flips into 2x-faster-generate mode.


In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

TEST_PROMPTS = [
    "turn off the bedroom light",
    "play lofi on the living room speaker",
    "is the main door open?",
]

for prompt in TEST_PROMPTS:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    print(f"\n--- USER: {prompt}")
    _ = model.generate(
        input_ids     = inputs,
        streamer      = TextStreamer(tokenizer, skip_prompt=True),
        max_new_tokens = 200,
        do_sample     = False,
    )


## 14. Save LoRA adapters locally (checkpoint)

Lightweight (<150 MB) snapshot of just the LoRA deltas. Useful for resuming or swapping adapters without a full 16-bit merge. The merged GGUF for Ollama is in the next step.


In [ ]:
LORA_DIR = f"{BUILD_TAG}-lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA adapter + tokenizer saved to ./{LORA_DIR}")


## 15. Export merged model to GGUF Q4_K_M for Ollama

`save_pretrained_gguf` merges the LoRA delta into the base model, quantises to Q4_K_M, and writes the result to `./{BUILD_TAG}/{BUILD_TAG}.Q4_K_M.gguf`. The final file is what you hand to `ollama create`.

Expected sizes: 2B ~1.5 GB, 4B ~2.7 GB.


In [ ]:
model.save_pretrained_gguf(
    BUILD_TAG,
    tokenizer,
    quantization_method = "q4_k_m",
)

GGUF_FILENAME = f"{BUILD_TAG}.Q4_K_M.gguf"
GGUF_PATH     = f"{BUILD_TAG}/{GGUF_FILENAME}"
print(f"GGUF at: {GGUF_PATH}")


## 16. (Optional) Push GGUF + merged 16-bit weights to HF Hub

Off by default -- the primary delivery channel is a local Ollama import of the GGUF file from step 15. Flip `PUSH_GGUF_TO_HUB = True` if you want a remote backup on the repo set in step 0 (`HUB_MODEL_REPO`). `PUSH_MERGED_TO_HUB` pushes full 16-bit weights (larger; only needed for vLLM/TGI serving later).


In [ ]:
PUSH_GGUF_TO_HUB   = False
PUSH_MERGED_TO_HUB = False

if PUSH_GGUF_TO_HUB:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HUB_MODEL_REPO, exist_ok=True, private=True, repo_type="model")
    api.upload_file(
        path_or_fileobj = GGUF_PATH,
        path_in_repo    = GGUF_FILENAME,
        repo_id         = HUB_MODEL_REPO,
        token           = HF_TOKEN,
    )
    print(f"GGUF pushed -> https://huggingface.co/{HUB_MODEL_REPO}/blob/main/{GGUF_FILENAME}")

if PUSH_MERGED_TO_HUB:
    merged_repo = HUB_MODEL_REPO.replace("-gguf", "-merged")
    model.push_to_hub_merged(
        merged_repo,
        tokenizer,
        save_method = "merged_16bit",
        token       = HF_TOKEN,
    )
    print(f"Merged 16-bit weights pushed -> https://huggingface.co/{merged_repo}")


## 17. Emit a ready-to-paste Ollama Modelfile

Written next to the GGUF as `{BUILD_TAG}.Modelfile`. Kept in sync with `finetuning/homebot_qwen3_5.Modelfile` in the repo (same fallback system prompt, same Qwen3 tool-calling template).


In [ ]:
# Kept in sync with `finetuning/homebot_qwen3_5.Modelfile` in the repo.
# Template is embedded inline so the notebook works on remote GPU boxes
# that don't have the rest of the repo checked out. Swap the FROM line
# and SYSTEM prompt at will -- it's just text.
import pathlib, re

MODELFILE_LINES = [
    '# HomeBot Qwen3.5-4B Fine-tune -- Ollama Modelfile',
    '#',
    '# After downloading `homebot-qwen3_5.Q4_K_M.gguf` from the Colab notebook',
    '# (unsloth_qwen3_5_4b_homebot.ipynb) into this directory, run:',
    '#',
    '#     ollama create homebot-qwen3_5 -f homebot_qwen3_5.Modelfile',
    '#',
    '# Then point the DeepAgent at the new model by setting:',
    '#',
    '#     MODEL=ollama:homebot-qwen3_5',
    '#',
    "# The DeepAgent's `_resolve_model` already handles the `ollama:` prefix.",
    '# -----------------------------------------------------------------------------',
    '',
    'FROM ./homebot-qwen3_5.Q4_K_M.gguf',
    '',
    '# Non-thinking Qwen3.5 sampling (matches Unsloth + Qwen docs).',
    '# Small Qwen3.5 models default to thinking-off, which is what we want for',
    '# deterministic tool calls.',
    'PARAMETER temperature 0.7',
    'PARAMETER top_p 0.8',
    'PARAMETER top_k 20',
    'PARAMETER repeat_penalty 1.05',
    'PARAMETER num_ctx 8192',
    'PARAMETER stop "<|im_end|>"',
    'PARAMETER stop "<|im_start|>"',
    'PARAMETER stop "<|endoftext|>"',
    '',
    "# Qwen3 ChatML template with tool_call + tool role support so Ollama's",
    '# chat endpoint can route our structured tool_calls through correctly.',
    'TEMPLATE """{{- if .Messages }}',
    '{{- range $i, $_ := .Messages }}',
    '{{- if eq .Role "system" }}<|im_start|>system',
    '{{ .Content }}<|im_end|>',
    '{{ else if eq .Role "user" }}<|im_start|>user',
    '{{ .Content }}<|im_end|>',
    '{{ else if eq .Role "assistant" }}<|im_start|>assistant',
    '{{- if .Content }}',
    '{{ .Content }}',
    '{{- end }}',
    '{{- if .ToolCalls }}',
    '<tool_call>',
    '{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}',
    '{{ end }}</tool_call>',
    '{{- end }}<|im_end|>',
    '{{ else if eq .Role "tool" }}<|im_start|>user',
    '<tool_response>',
    '{{ .Content }}',
    '</tool_response><|im_end|>',
    '{{ end }}',
    '{{- end }}<|im_start|>assistant',
    '{{ end }}"""',
    '',
    '# Fallback SYSTEM prompt used only when the DeepAgent does not supply one',
    '# (e.g. direct `ollama run homebot-qwen3_5`). At runtime inside the homebot',
    '# the DeepAgent always injects the authoritative get_system_prompt() from',
    '# deepagent/agent.py, which overrides this fallback.',
    'SYSTEM """You are HomeBotAI, an intelligent smart-home assistant powered by Home Assistant.',
    'The home is in India (IST timezone). Resident: Kanak.',
    '',
    'You have access to tools for:',
    '- Home Assistant device control (ha_call_service, ha_get_states, ha_search_entities)',
    '- Media management (sonarr_*, radarr_*, jellyfin_*, jellyseerr_*, prowlarr_*, transmission_*)',
    '- Network admin (deco_list_clients, deco_list_mesh_nodes, deco_reboot_nodes, deco_reservation_help)',
    '- Obsidian vault + persistent memory (obsidian_*, memory_*)',
    '- Link processing (process_and_save_link)',
    '- Interactive choices (offer_choices -- tap-able buttons; end your turn after calling it)',
    '- Shell execution (execute)',
    '',
    'Rules:',
    '1. Be efficient with tool calls -- 1-3 targeted calls over exhaustive searching.',
    '2. Always provide a short natural-language summary after tool calls.',
    '3. Use colloquial names in replies (e.g. "the purifier"), never raw entity_ids.',
    '4. For short ordinal replies like "3" or "the second one", resolve against your',
    '   previous message.',
    '5. Confirm actions in one line and stop -- no filler tails, no second-guessing.',
    '6. Synthesize redundant sensor readings (within ~1C / 5%RH) into one value instead',
    '   of dumping raw lists.',
    '"""',
]
MODELFILE_TEMPLATE = "\n".join(MODELFILE_LINES) + "\n"

# Rewrite the FROM line so it points at the GGUF we just produced.
MODELFILE_TEMPLATE = re.sub(
    r"^FROM .*$",
    f"FROM ./{BUILD_TAG}.Q4_K_M.gguf",
    MODELFILE_TEMPLATE,
    count=1,
    flags=re.M,
)

MODELFILE_PATH = f"{BUILD_TAG}.Modelfile"
pathlib.Path(MODELFILE_PATH).write_text(MODELFILE_TEMPLATE)

print(f"Modelfile written to ./{MODELFILE_PATH}")
print()
print("Next steps:")
print(f"  1. scp {GGUF_PATH} {MODELFILE_PATH} user@your-mac:~/homebot/")
print(f"  2. ollama create {BUILD_TAG} -f ./{MODELFILE_PATH}")
print(f"  3. ollama run {BUILD_TAG}")
print(f"  4. In deepagent/.env set:  MODEL=ollama:{BUILD_TAG}")


---

**Troubleshooting**

- *`No CUDA GPU visible`*: check `nvidia-smi` from the same shell that launched Jupyter. If it reports fine but Python doesn't see it, the kernel was started before CUDA libs loaded -- restart the kernel.
- *`torch.cuda.is_bf16_supported() == False`*: pre-Ampere GPU (Turing / Volta / Pascal). This notebook auto-falls-back to fp16, but performance is noticeably lower; consider using the Colab T4 notebook instead, which is built around the fp16 path.
- *OOM during training (step 11)*: drop `BATCH_SIZE` in step 0 (or set it explicitly to a smaller value like 2). Alternatively raise `GRAD_ACCUM` from 1 to 2/4 to keep effective batch size while reducing per-step VRAM.
- *Flash Attention missing / warnings like "falling back to xformers"*: add `pip install -q flash-attn --no-build-isolation` in step 1 for a ~15-20% speedup on Ampere+. Safe to skip; xformers is fine.
- *GGUF export fails with "unsupported architecture"*: Unsloth's `save_pretrained_gguf` needs the merged 16-bit weights. Retry once; if it fails again bypass with `model.save_pretrained_merged(...)` + `llama.cpp/convert-hf-to-gguf.py`.
- *Low eval loss but bad tool calls*: inspect the rendered example in step 7 -- make sure tool messages show up as `<|im_start|>tool ...`.
- *Tool calls fire but summary is verbose/robotic*: model is over-fitting the simulator style. Increase `REAL_OVERSAMPLE` in step 0 (try 6-8) or pull more real Telegram traces locally and re-merge.
- *Model only replies in natural language and never calls a tool*: step 9 (`train_on_responses_only`) was skipped. Without it the model learns to reproduce the system prompt, crowding out tool-call patterns.

**Iteration recipe (same as Colab notebook)**

1. `scp` the GGUF + Modelfile to the Mac, `ollama create`, try 5-10 real Telegram-style messages.
2. If tool arguments are wrong, the dataset is too small for that skill. Run `./run_pipeline.sh generate` + `simulate`, re-merge, push, re-train.
3. If reply style is off but tool calls are right, oversample real data more aggressively (step 0 `REAL_OVERSAMPLE=6` or `8`) rather than adding more synthetic rows.
